# 🔵 Kernel Ridge Regression — LOOCV
**51 amostras × 26 índices espectrais**

## 1. Bibliotecas

In [ ]:
!pip install openpyxl --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
SEED = 42; np.random.seed(SEED)

def calc_metrics(y_true, y_pred, model_name='Modelo'):
    r2    = r2_score(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    nrmse = rmse / (y_true.max() - y_true.min()) * 100
    bias  = np.mean(y_pred - y_true)
    mape  = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rpd   = y_true.std() / rmse
    mu_t, mu_p = y_true.mean(), y_pred.mean()
    s2_t = np.var(y_true); s2_p = np.var(y_pred)
    s_tp = np.mean((y_true - mu_t) * (y_pred - mu_p))
    ccc  = (2 * s_tp) / (s2_t + s2_p + (mu_t - mu_p) ** 2)
    print(f'\n=== {model_name} — LOOCV ===')
    print(f'  R²={r2:.4f}  CCC={ccc:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}')
    print(f'  NRMSE={nrmse:.2f}%  MAPE={mape:.2f}%  Bias={bias:.4f}  RPD={rpd:.4f}')
    return {'Modelo': model_name,
            'R2': round(r2,4), 'CCC': round(ccc,4), 'RMSE': round(rmse,4),
            'MAE': round(mae,4), 'NRMSE(%)': round(nrmse,2),
            'MAPE(%)': round(mape,2), 'Bias': round(bias,4), 'RPD': round(rpd,4)}

print('✅ Bibliotecas carregadas!')
from sklearn.kernel_ridge import KernelRidge

## 2. Carregar Dados

In [ ]:
import os, io
IN_COLAB = False
try:
    import google.colab; IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import files
    print("📂 Selecione o arquivo Excel ou CSV:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    ext = fname.split('.')[-1].lower()
    df = pd.read_excel(io.BytesIO(uploaded[fname])) if ext in ['xlsx','xls'] else pd.read_csv(io.BytesIO(uploaded[fname]))
else:
    FILE_PATH = 'só_indices.xlsx'  # ⚠️ ajuste se necessário
    assert os.path.exists(FILE_PATH), f"Arquivo não encontrado: {FILE_PATH}"
    df = pd.read_excel(FILE_PATH)

TARGET    = 'Biomass_g'
DROP_COLS = ['Flight_ID']
FEATURES  = [c for c in df.columns if c not in [TARGET] + DROP_COLS]
df = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)
X  = df[FEATURES].values
y  = df[TARGET].values
print(f'✅ Amostras: {len(y)}  |  Features: {len(FEATURES)}')
df.head()

## 3. Definir Modelo

In [ ]:
MODEL_NAME  = 'KRR'
MODEL_COLOR = '#4169E1'

pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('model',  KernelRidge(kernel='rbf', alpha=0.1, gamma=0.1))
])

## 4. LOOCV — Leave-One-Out Cross-Validation

In [ ]:
print(f'▶ Rodando LOOCV ({len(y)} iterações)...')
loo = LeaveOneOut()
y_pred = cross_val_predict(pipe, X, y, cv=loo)
metrics_dict = calc_metrics(y, y_pred, MODEL_NAME)

## 5. Gráfico Observado vs Predito + Resíduos

In [ ]:
# ── Observado vs Predito + Resíduos ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plt.rcParams['font.family'] = 'Times New Roman'

lim = [min(y.min(), y_pred.min()) * 0.9, max(y.max(), y_pred.max()) * 1.1]

ax = axes[0]
ax.scatter(y, y_pred, color=MODEL_COLOR, edgecolors='white', s=60, alpha=0.85, zorder=3)
ax.plot(lim, lim, 'k--', lw=1.5, label='1:1', zorder=2)
m_fit, b_fit = np.polyfit(y, y_pred, 1)
xfit = np.linspace(lim[0], lim[1], 100)
ax.plot(xfit, m_fit * xfit + b_fit, 'r-', lw=2, alpha=0.85, label='Ajuste', zorder=4)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Observado (g)', fontsize=13)
ax.set_ylabel('Predito (g)', fontsize=13)
ax.set_title(f'{MODEL_NAME} — LOOCV\nR²={metrics_dict["R2"]:.3f}  CCC={metrics_dict["CCC"]:.3f}  RMSE={metrics_dict["RMSE"]:.2f}g',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.3)

ax2 = axes[1]
residuals = y - y_pred
ax2.scatter(y_pred, residuals, color=MODEL_COLOR, edgecolors='white', s=60, alpha=0.85)
ax2.axhline(0, color='red', linestyle='--', lw=1.5)
ax2.set_xlabel('Predito (g)', fontsize=13)
ax2.set_ylabel('Resíduo (g)', fontsize=13)
ax2.set_title('Resíduos', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{MODEL_NAME}_LOOCV_obs_pred.png', dpi=180, bbox_inches='tight')
plt.show()
print("✅ Gráfico salvo.")

## 6. Importância das Features

In [ ]:
# ── Importância das Features (Permutation Importance) ────────────────────────
from sklearn.inspection import permutation_importance
pipe.fit(X, y)
result = permutation_importance(pipe, X, y, n_repeats=20, random_state=SEED, n_jobs=-1)
imp_vals = pd.Series(result.importances_mean, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, max(5, len(FEATURES)*0.38)))
plt.rcParams['font.family'] = 'Times New Roman'
ax.barh(imp_vals.index, imp_vals.values, color=MODEL_COLOR, edgecolor='white')
ax.set_xlabel('Importância (redução média no R²)', fontsize=13)
ax.set_title(f'{MODEL_NAME} — Importância das Features', fontsize=13, fontweight='bold')
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig(f'{MODEL_NAME}_LOOCV_importancia.png', dpi=180, bbox_inches='tight')
plt.show()
print("✅ Gráfico salvo.")

## 7. Exportar Resultados

In [ ]:
# ── Salvar métricas e predições ──────────────────────────────────────────────
pd.DataFrame([metrics_dict]).to_csv(f'{MODEL_NAME}_LOOCV_metricas.csv', index=False)
pd.DataFrame({'y_obs': y, 'y_pred': y_pred}).to_csv(f'{MODEL_NAME}_LOOCV_predicoes.csv', index=False)
print(pd.DataFrame([metrics_dict]).to_string(index=False))

import zipfile
files_to_zip = [f for f in os.listdir('.') if MODEL_NAME in f and (f.endswith('.png') or f.endswith('.csv'))]
zip_name = f'{MODEL_NAME}_LOOCV_resultados.zip'
with zipfile.ZipFile(zip_name, 'w') as zf:
    for f in files_to_zip: zf.write(f); print(f'  + {f}')
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(zip_name)
else:
    print(f'✅ Salvo na pasta do notebook.')